In [0]:
spark

### TYPES OF TABLES - Managed Table(Python)

In [0]:
#-- I have a file in my Databricks workspace and below iam reading the file and creating a managed delta table.

df = spark.read.format("csv").options(header='true', inferSchema='true').load("/Volumes/workspace/default/inbound/credit_card_transcations.csv")

df.write.mode("overwrite").format("delta").saveAsTable("sharath_lakeflow.different_tables.credit_card")



In [0]:
#-- Writing the data and saving it as a delta table with some transformations

from pyspark.sql.functions import *
from pyspark.sql.types import *

df = spark.read.table("sharath_lakeflow.different_tables.credit_card")

df = df.withColumn("city",lower(df.city))

df.write.mode("overwrite").format("delta").saveAsTable("sharath_lakeflow.different_tables.credit_card")

In [0]:
# -- saving the data as a parquet file in s3 bucket
from pyspark.sql.functions import *
from pyspark.sql.types import *

df = spark.read.table("sharath_lakeflow.different_tables.credit_card")

df = df.withColumn("city",lower(df.city))

df.write.mode("overwrite").format("parquet").save("s3://inboundfiless/Outbound/")

In [0]:
%sql
Describe table extended sharath_lakeflow.different_tables.credit_card;


In [0]:
%sql
describe detail sharath_lakeflow.different_tables.credit_card;

In [0]:
%sql
describe history sharath_lakeflow.different_tables.credit_card;

### Creating external table (using sql/pyspark)

In [0]:
#iam reading the file from s3 bucket and creating the dataframe the loading into a external table

df = spark.read.format("parquet").load("s3://inboundfiless/products.parquet")

df.write.mode("overwrite").format("delta").option("path", "s3://inboundfiless/products.parquet").saveAsTable("sharath_lakeflow.different_tables.product_external")

In [0]:
%sql
describe table extended sharath_lakeflow.different_tables.product_external;

In [0]:
%sql 
--using sql syntax to create external table.
create table sharath_lakeflow.different_tables.region
using PARQUET
location "s3://inboundfiless/regions.parquet"

In [0]:
%sql
describe table extended sharath_lakeflow.different_tables.region;

### Data Ingestion techniques

In [0]:
%sql

create table sharath_lakeflow.different_tables.orders_external1
USING parquet
location 's3://inboundfiless/orders.parquet'

In [0]:
%sql
describe table extended sharath_lakeflow.different_tables.orders_external;

In [0]:
df = spark.read.table("sharath_lakeflow.different_tables.region")
df.write.mode("overwrite").format("delta").option("path","s3://inboundfiless/region").save()

### Creating Streaming Table

In [0]:
%sql
use catalog sharath_lakeflow;
use schema different_tables;

In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE iris
as select * from stream cloud_files("s3://inboundfiless/Inbounddd/","json",
map('cloudFiles.inferColumnTypes','true','cloudFiles.schemaLocation','s3://inboundfiless/Inbounddd'));

In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE iris
as select * from stream(sharath_lakeflow.different_tables.region);